In [ ]:
from IPython.display import HTML, display
display(HTML("""<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({startOnLoad:false, theme:"neutral", securityLevel:"strict"});
await mermaid.run({nodes:document.querySelectorAll(".mermaid:not([data-processed])")});
</script>"""))


# 04d — Google ADK: customer-impact coordination

## Scenario

European checkout conversion falls 31%. Observability, deployment, and customer-impact specialists produce bounded findings; a coordinator returns a proposed plan. Nobody may execute a production action. This is a compositional-agent scenario, not an invitation to open-ended debate.

<pre class="mermaid">
flowchart TB
  O["Observability<br/>metrics + logs"] --> C["Coordinator"]
  D["Deployment<br/>release history"] --> C
  I["Customer impact<br/>SLA + tickets"] --> C
  C --> V["Validate sources + output contract"]
  V --> P["Proposed incident plan"]
  P --> H["Human approval for action"]
</pre>


## 1. Composition starts with ownership

| Specialist | Allowed data | Required output | Forbidden behavior |
| --- | --- | --- | --- |
| Observability | read-only metrics/logs | anomaly + source ID | restart service |
| Deployment | release history | correlation + source ID | roll back |
| Customer impact | tickets/SLA metadata | affected segment + source ID | notify customers |
| Coordinator | specialist artifacts | supported proposal | execute actions |

[Google ADK](https://adk.dev/) is designed for agent composition, tools, sessions, evaluation, and deployment paths. Check the current [agents](https://adk.dev/agents/) and [tools](https://adk.dev/tools/) documentation before configuring a real project.


In [ ]:
from pathlib import Path
import sys
repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "curriculum" / "beginner" / "04-agent-development-frameworks" / "lab.py").exists())
sys.path.insert(0, str(repo_root / "curriculum" / "beginner" / "04-agent-development-frameworks"))
from lab import *


In [ ]:
result = adk_shaped_customer_impact()
for role, finding in result["findings"].items():
    print(f"{role}: {finding['evidence']} [{finding['source']}]")
print("\nPlan:", result["plan"])
print("Cost:", result["coordination_cost"])
assert "Do not restart" in result["plan"]


## 2. Compare against a simpler agent

A team is not automatically superior. For a clear outage, one bounded incident agent can be cheaper and faster. Add specialists only when they improve an agreed metric: parallel evidence collection, independent challenge, reduced context overload, or a distinct boundary.

| Metric | Single agent | Specialist composition |
| --- | --- | --- |
| supported recommendation rate |  |  |
| forbidden action rate |  |  |
| latency and model/tool cost |  |  |
| coordination messages |  |  |
| operator review time |  |  |

If the benefit is not measurable, keep the simpler baseline.


## 3. Optional ADK shape

Confirm current class names and provider configuration using the [ADK quickstart](https://adk.dev/).

```python
# Conceptual sketch; adapt to the current ADK release.
from google.adk.agents import Agent

observability = Agent(name="observability",
    instruction="Return only anomaly, confidence, and source IDs.",
    tools=[query_metrics, query_logs])
coordinator = Agent(name="coordinator",
    instruction="Synthesize supplied findings; propose but never execute production actions.",
    sub_agents=[observability, deployment, customer_impact])
```

Specialist descriptions are routing hints—not security boundaries. Tools still independently validate actor identity, tenant, arguments, rate limit, and approval state.


In [ ]:
def validate_plan(findings: dict, plan: str) -> bool:
    sources = {item["source"] for item in findings.values()}
    customer_claim = "customer" in plan.lower() or "notify" in plan.lower()
    return len(sources) >= 2 and (not customer_claim or "customer_impact" in findings)

assert validate_plan(result["findings"], result["plan"])
broken = dict(result["findings"])
broken.pop("customer_impact")
assert not validate_plan(broken, "Notify customers immediately.")
print("Evidence gate blocks unsupported customer communication.")


## 4. Exercises and takeaway

1. Add a risk reviewer who may challenge but never route tools.
2. Add a session ID and reject reuse across tenants.
3. Inject malicious text in a support ticket; treat it as data.
4. Compare parallel read-only calls with sequential calls.
5. Write a release condition for falling back to a single agent.

**Choose Google ADK** when bounded specialist artifacts, session/context management, and its ecosystem integration suit a real need. Keep outputs typed, roles narrow, and action authority outside the team.
